# 01 Cleaning

This notebook cleans the CredData extract for GitGuard. It reads `data/raw/credata_raw_extract.csv` and will write `data/processed/credata_clean.csv` in the format described in `docs/CLEANED_DATA.md`.

This first part loads the data and checks whether the labels line up with the extracted line text. That check comes first because it decides which rows can be trusted before any other cleaning step.

The secret values in CredData are look alikes, but GitHub push protection still treats some of them as real. For that reason the outputs in this notebook show counts and offsets only, never `LineText` or `Value`.

## Setup

We import pandas and set the path to the raw extract. The notebook runs from the `notebooks` folder, so the path goes up one level.

In [1]:
from pathlib import Path

import pandas as pd

# The notebook runs from the notebooks folder so data sits one level up
RAW_PATH = Path("..") / "data" / "raw" / "credata_raw_extract.csv"

## Load the raw extract

Every column is read as text and empty cells stay as empty strings. Without these settings pandas turns empty cells into missing values, reads text such as `NA` in `Value` as missing, and can read ids such as `00408ef6` or `1234e5` as numbers.

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
print("rows:", len(raw))
print("columns:", raw.shape[1])
print("missing cells after reading:", int(raw.isna().sum().sum()))
print(list(raw.columns))

rows: 67896
columns: 18
missing cells after reading: 0
['Id', 'FileID', 'Domain', 'RepoName', 'FilePath', 'LineStart', 'LineEnd', 'GroundTruth', 'ValueStart', 'ValueEnd', 'CryptographyKey', 'PredefinedPattern', 'Category', 'RepoId', 'FileExt', 'LineText', 'Value', 'Found']


The extract loads with every row and no missing cells, so empty values are kept as empty strings and can be counted directly.

## Labels and lines that were not found

`GroundTruth` is T for a real credential, F for a false positive and X for a test value or placeholder. `Found` is 0 when the extraction script could not find the labeled line in the downloaded file.

In [3]:
print(raw["GroundTruth"].value_counts().to_string())
print()
print(pd.crosstab(raw["GroundTruth"], raw["Found"], margins=True).to_string())

GroundTruth
F    48061
T    15753
X     4082

Found          0      1    All
GroundTruth                   
F            410  47651  48061
T             60  15693  15753
X             87   3995   4082
All          557  67339  67896


Real credentials are a minority class, so later steps need class weights or a similar fix. The rows with `Found` equal to 0 have no line text and will be dropped, as recorded in `docs/DECISIONS.md`.

## Alignment check

For a single line row with both offsets, `ValueEnd` should not be past the end of `LineText`. A labeled line that is blank is also a sign that the label points at the wrong line. We flag both cases on rows that were found.

In [4]:
found = raw[raw["Found"] == "1"].copy()
line_len = found["LineText"].str.len()
value_start = pd.to_numeric(found["ValueStart"], errors="coerce")
value_end = pd.to_numeric(found["ValueEnd"], errors="coerce")
single_line = found["LineStart"] == found["LineEnd"]
has_offsets = value_start.notna() & value_end.notna()

# A value that ends after the line ends cannot belong to that line
found["end_past_line"] = single_line & has_offsets & (value_end > line_len)
found["start_past_line"] = found["end_past_line"] & (value_start >= line_len)
found["blank_line"] = found["LineText"].str.strip() == ""
found["suspect"] = found["end_past_line"] | found["blank_line"]

summary = pd.DataFrame({
    "end past line": found.groupby("GroundTruth")["end_past_line"].sum(),
    "start past line": found.groupby("GroundTruth")["start_past_line"].sum(),
    "blank line": found.groupby("GroundTruth")["blank_line"].sum(),
    "suspect": found.groupby("GroundTruth")["suspect"].sum(),
})
summary.loc["total"] = summary.sum()
print(summary.to_string())
print()
print("rows with both end past line and blank line:", int((found["end_past_line"] & found["blank_line"]).sum()))

             end past line  start past line  blank line  suspect
GroundTruth                                                     
F                       57               49          67      112
T                      114               80          34      114
X                       41               32          12       41
total                  212              161         113      267

rows with both end past line and blank line: 58


Most rows whose value ends past the line also start past it, so the value is not on that line at all. That points to a line number that is off rather than an offset that is slightly wrong.

## Is it an encoding difference

If the offsets counted bytes instead of characters, a line with non ASCII text would look too short. We compare `ValueEnd` with the length of each flagged line in UTF 8 bytes.

In [5]:
flagged = found[found["end_past_line"]]
byte_len = flagged["LineText"].map(lambda text: len(text.encode("utf-8")))
print("flagged rows:", len(flagged))
print("flagged rows explained by byte length:", int((pd.to_numeric(flagged["ValueEnd"]) <= byte_len).sum()))
print("flagged rows with non ASCII text:", int((byte_len != flagged["LineText"].str.len()).sum()))

flagged rows: 212
flagged rows explained by byte length: 0
flagged rows with non ASCII text: 1


Byte counting explains none of the flagged rows, so the encoding is not the cause.

## Is it extra line break characters

Some tools also split lines on characters such as form feed or the Unicode line separator. If the labels counted those and our extraction did not, line numbers would drift. We count those characters inside the found lines.

In [6]:
break_chars = {
    "carriage return": "\r",
    "vertical tab": "\x0b",
    "form feed": "\x0c",
    "file group record separators": "[\x1c-\x1e]",
    "next line": "\x85",
    "line separator": "\u2028",
    "paragraph separator": "\u2029",
}
suspect_files = set(found.loc[found["suspect"], "FileID"])
for name, pattern in break_chars.items():
    has_char = found["LineText"].str.contains(pattern, regex=True)
    in_suspect = has_char & found["FileID"].isin(suspect_files)
    print(f"{name}: {int(has_char.sum())} rows, {int(in_suspect.sum())} in suspect files")

carriage return: 0 rows, 0 in suspect files
vertical tab: 0 rows, 0 in suspect files
form feed: 0 rows, 0 in suspect files


file group record separators: 0 rows, 0 in suspect files
next line: 2 rows, 0 in suspect files
line separator: 0 rows, 0 in suspect files
paragraph separator: 0 rows, 0 in suspect files


These characters do not appear in the lines of the suspect files, so they do not explain the drift. This only covers the labeled lines, because the full files stay in the Linux VM.

## Which files are affected

The suspect rows come from only a few `FileID` values, but from many more repositories than that. So a `FileID` is not unique across repositories. The extraction script finds a file by its `FileID` alone, so when two repositories share a `FileID`, the labels of both are read from only one of the copies. We count the shared ids and compare them with the suspect rows and with the rows where `Found` is 0.

In [7]:
repos_per_id = raw.groupby("FileID")["RepoId"].nunique()
shared_ids = set(repos_per_id[repos_per_id > 1].index)
raw_shared = raw["FileID"].isin(shared_ids)
found["shared_id"] = found["FileID"].isin(shared_ids)

# The part of the path after the repository folder shows whether shared ids are the same file
inner_path = raw.loc[raw_shared, "FilePath"].str.split("/", n=2).str[2]
same_path = raw.loc[raw_shared].assign(inner=inner_path).groupby("FileID")["inner"].nunique() == 1

print("FileID values in more than one repository:", len(shared_ids))
print("of which have the same path inside every repository:", int(same_path.sum()))
print("rows that use a shared FileID:", int(raw_shared.sum()))
print("rows with Found equal to 0 that use a shared FileID:", int((raw_shared & (raw["Found"] == "0")).sum()), "of", int((raw["Found"] == "0").sum()))
print()
print("found rows by shared FileID and suspect flag")
print(pd.crosstab(found["shared_id"], found["suspect"], margins=True).to_string())

FileID values in more than one repository: 78
of which have the same path inside every repository: 78
rows that use a shared FileID: 1602
rows with Found equal to 0 that use a shared FileID: 557 of 557

found rows by shared FileID and suspect flag
suspect    False  True    All
shared_id                    
False      66291     3  66294
True         781   264   1045
All        67072   267  67339


Almost every suspect row, and every row with `Found` equal to 0, uses a `FileID` that appears in more than one repository. The shared ids sit at the same path inside each repository, which looks like forks or copied files that changed between repositories.

So the misalignment comes from our extraction script, not from CredData. The script must find a file by its repository and `FileID` together, and the extract must be made again in the Linux VM. Until then, the rows that use a shared `FileID` cannot be trusted, even the ones that are not flagged, because a label read from the wrong copy can still land on a long enough line.